<a href="https://colab.research.google.com/github/WendelLeander/PhishEvade/blob/main/feature_extraction_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lexical and Obfuscation-Aware Feature Extraction
### Obfuscation-Aware Feature Engineering for Phishing URL Detection

This notebook implements the **Feature Extraction Module** described in Chapter 4 of the study proposal, split into the two milestones:

| Milestone | Module | Feature level | Features |
|---|---|---|---|
| **3** | Traditional Lexical Feature Extraction | **Level 1** (shallow lexical criteria) | 3 |
| **4** | Obfuscation-Aware Feature Engineering | **Level 2** (deep obfuscation-aware criteria) | 10 |

Together they produce the **13 engineered attributes** of Table 4.1, appended to the cleaned dataset as numeric columns.
Scaling (Min-Max / Z-Score) belongs to the next stage and is intentionally **not** applied here, so the saved features stay raw and interpretable (and scalers can later be fitted on the training split only, avoiding leakage).

#### Tools used (Section 1.6.2 of the proposal)

| Tool | Role in this notebook |
|---|---|
| **pandas / NumPy** | Data frames, vectorised preprocessing, numeric summaries |
| **urllib.parse** (`urlparse`, `parse_qsl`) | Splitting each URL into network location, path and query |
| **tldextract** | Isolating the subdomain string (offline, bundled suffix list) |
| **statistics / re / math** | Token variance, regex tokenisation, Shannon entropy |
| **Unfurl** (`dfir-unfurl`) | Recursive percent-/base64-decoding of the **query component** for the three query-level features |
| **psutil** | Wall-clock time, CPU time/utilisation and **peak memory** of each extraction phase (Computational Resource Metrics, Section 3.8.5) |

#### The 13 features (Table 4.1)

| # | Feature | Level | Procedure (Feature Extraction Execution) | Type |
|---|---|---|---|---|
| 1 | `url_length` | 1 | Lexical Counts | int |
| 2 | `domain_length` | 1 | Lexical Counts | int |
| 3 | `path_length` | 1 | Lexical Counts | int |
| 4 | `subdomain_count` | 2 | Lexical Counts (tldextract) | int |
| 5 | `path_depth` | 2 | Lexical Counts | int |
| 6 | `query_params_count` | 2 | Lexical Counts | int |
| 7 | `digit_ratio` | 2 | Ratio Generation | float |
| 8 | `special_char_ratio` | 2 | Ratio Generation | float |
| 9 | `token_variance` | 2 | Complexity Derivation | float |
| 10 | `domain_entropy` | 2 | Entropy Calculation | float |
| 11 | `redirect_key_flag` | 2 | Query Analysis (Unfurl) | binary |
| 12 | `nested_url_flag` | 2 | Query Analysis (Unfurl) | binary |
| 13 | `query_encoding_ratio` | 2 | Query Analysis (raw query) | float |

**Installation** (once): `pip install pandas numpy tldextract psutil dfir-unfurl`


## 0. Setup and Input Data

The input is `Final Dataset.csv` (columns `url`, `label`; `0` = benign, `1` = malicious). So that this notebook is self-contained and always works on the latest dataset, the **cleaning logic of Section 4.4.1** is re-applied here in compact form, in the order given in the proposal:

1. `dropna()` on corrupted rows, then `drop_duplicates()` on the `url` column
2. Lowercase every string
3. A sequential pair of regular expressions: strip `http://` / `https://`, then strip `www.`
4. Down-sample the majority class to the minority count (strict 1:1 balance)

> **Why `www.` is stripped here:** `tldextract` treats `www` as a *subdomain*. If it were left in, every `www.`-prefixed URL would get `subdomain_count = 1` (and `domain_length` +4) purely from a formatting habit, not from obfuscation. Set `STRIP_WWW = False` to reproduce the narrower protocol-only standardization of Section 3.7.1.


In [11]:
!pip install tldextract
import os, re, math, time, threading, statistics, logging
from collections import Counter, defaultdict
from datetime import datetime
from urllib.parse import urlparse, parse_qsl, unquote


import numpy as np
import pandas as pd
import psutil
import tldextract

try:
    from unfurl import core as unfurl_core
    UNFURL_AVAILABLE = True
    logging.getLogger("unfurl").setLevel(logging.ERROR)
except ImportError:
    UNFURL_AVAILABLE = False

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)

DATA_PATH = 'https://raw.githubusercontent.com/WendelLeander/PhishEvade/refs/heads/main/Final%20Dataset.csv'
STRIP_WWW = True
RANDOM_SEED = 42

print("pandas", pd.__version__, "| numpy", np.__version__, "| psutil", psutil.__version__)
print("Unfurl available:", UNFURL_AVAILABLE)
if not UNFURL_AVAILABLE:
    print("  -> 'dfir-unfurl' not installed; Section 4.5 will use a built-in recursive decoder instead.\n"
          "     Install it (pip install dfir-unfurl) to follow the proposal exactly.")

pandas 2.2.3 | numpy 2.1.3 | psutil 5.9.5
Unfurl available: False
  -> 'dfir-unfurl' not installed; Section 4.5 will use a built-in recursive decoder instead.
     Install it (pip install dfir-unfurl) to follow the proposal exactly.


## 1. Resource Instrumentation with `psutil`

Section 3.8.5 requires the pipeline to log **wall-clock timestamps** at the start and end of feature extraction and to capture **peak memory usage via psutil** for each phase, so the cost of the obfuscation-aware feature set can be compared against the traditional baseline.

`ResourceProfiler` is a small context manager wrapped around every extraction phase below. It records:

- **Start / end wall-clock time** and duration (`time.perf_counter`)
- **CPU time** (user + system) and **average CPU utilisation** from `psutil.Process`
- **Peak resident memory (RSS)**, sampled every 10 ms by a background thread (psutil has no cross-platform "peak" counter), and the increase over the starting level
- **Milliseconds per URL**, to make phases comparable


In [12]:
RESOURCE_LOG = []   # one record per profiled phase

class ResourceProfiler:
    '''Context manager that logs wall time, CPU time/utilisation and peak memory of a phase.'''

    def __init__(self, phase, n_urls, n_features, interval=0.01):
        self.phase, self.n_urls, self.n_features, self.interval = phase, n_urls, n_features, interval

    def __enter__(self):
        self.proc = psutil.Process(os.getpid())
        self.proc.cpu_percent(None)                      # prime psutil's CPU-utilisation counter
        self.cpu0 = self.proc.cpu_times()
        self.rss0 = self.proc.memory_info().rss
        self.peak = self.rss0
        self._stop = threading.Event()
        self._thread = threading.Thread(target=self._poll, daemon=True)
        self.start_dt = datetime.now()
        self.t0 = time.perf_counter()
        self._thread.start()
        return self

    def _poll(self):
        while not self._stop.is_set():
            self.peak = max(self.peak, self.proc.memory_info().rss)
            time.sleep(self.interval)

    def __exit__(self, *exc):
        wall = time.perf_counter() - self.t0
        self._stop.set(); self._thread.join()
        self.peak = max(self.peak, self.proc.memory_info().rss)
        cpu1 = self.proc.cpu_times()
        cpu_s = (cpu1.user + cpu1.system) - (self.cpu0.user + self.cpu0.system)
        RESOURCE_LOG.append({
            "phase": self.phase,
            "features": self.n_features,
            "start": self.start_dt.strftime("%H:%M:%S"),
            "end": datetime.now().strftime("%H:%M:%S"),
            "wall_s": wall,
            "cpu_time_s": cpu_s,
            "cpu_util_%": self.proc.cpu_percent(None),
            "peak_rss_mb": self.peak / 2**20,
            "delta_rss_mb": (self.peak - self.rss0) / 2**20,
            "ms_per_url": wall / max(self.n_urls, 1) * 1000,
        })
        return False

print("ResourceProfiler ready (", psutil.cpu_count(logical=True), "logical CPUs,",
      round(psutil.virtual_memory().total / 2**30, 1), "GB RAM )")

ResourceProfiler ready ( 2 logical CPUs, 12.7 GB RAM )


---
# Milestone 3: Traditional (Level 1) Lexical Feature Extraction Module

Level 1 captures only the **basic absolute character lengths** of a web address: `url_length`, `domain_length` and `path_length`. They give a foundational geometric baseline, but a model that relies on them alone overfits to rigid length thresholds and fails when attackers deliberately shorten or pad paths. That weakness is exactly what Level 2 (Milestone 4) addresses.

## 3.1 Lexical Parsing Submodule

Before any length can be measured per component, every cleaned URL is deconstructed into its hierarchical components (RFC 3986). Following the *Feature Extraction Execution* procedure:

1. A **dummy `http://` scheme is appended** so that `urlparse` correctly identifies the network-location boundary (the cleaned strings no longer carry a scheme).
2. **domain** = the network location (`netloc`) returned by `urlparse`, i.e. the host portion *including any subdomains* (so entropy can expose randomly generated subdomains).
3. **path** and **query** come straight from `urlparse`.
4. **tldextract** separately isolates the **subdomain** string (labels above the registered domain). It is used in offline mode with its bundled public-suffix snapshot, so no network access is needed.


In [13]:
_TLD = tldextract.TLDExtract(suffix_list_urls=(), cache_dir=None)   # offline: bundled suffix list
PARSE_FALLBACKS = 0

def parse_url(url: str) -> dict:
    '''Lexical Parsing Submodule: split a cleaned URL into domain, subdomain, path and query.'''
    global PARSE_FALLBACKS
    try:
        parts = urlparse("http://" + url)             # dummy scheme so urlparse finds the netloc
        host = parts.hostname or ""
        return {"domain": parts.netloc, "subdomain": _TLD(host).subdomain,
                "path": parts.path, "query": parts.query}
    except ValueError:                                # malformed authority (e.g. broken IPv6 bracket)
        PARSE_FALLBACKS += 1
        host, _, rest = url.partition("/")
        path, _, query = (("/" + rest) if rest else "").partition("?")
        return {"domain": host, "subdomain": "", "path": path, "query": query}

# anatomy of one parsed URL
example = df.loc[df["url"].str.contains(r"\?.+=", regex=True)].iloc[0]["url"]
print("URL      :", example)
for k, v in parse_url(example).items():
    print(f"{k:<10}:", v)

URL      : builder.aws.com/?trk=8fe18e89-d22f-462a-8d9a-4fa301e62e02&sc_channel=ps
domain    : builder.aws.com
subdomain : builder
path      : /
query     : trk=8fe18e89-d22f-462a-8d9a-4fa301e62e02&sc_channel=ps


In [14]:
with ResourceProfiler("Lexical parsing (shared by Levels 1 & 2)", len(df), 0):
    components = [parse_url(u) for u in df["url"]]

print(f"Parsed {len(components):,} URLs | fallback parses: {PARSE_FALLBACKS}")
pd.concat([df["url"], pd.DataFrame(components)], axis=1).sample(5, random_state=RANDOM_SEED)

Parsed 10,000 URLs | fallback parses: 0


,url,domain,subdomain,path,query
6252,bit.ly/4itvu9r,bit.ly,,/4itvu9r,
4684,docs.google.com/presentation/d/e/2pacx-1vqqok9dyd52pv6qhjyydlupmze...,docs.google.com,docs,/presentation/d/e/2pacx-1vqqok9dyd52pv6qhjyydlupmzezobswxapnjm-lrv...,start=false&amp;loop=false&amp;delayms=3000
1731,repost.aws/reviews/,repost.aws,,/reviews/,
4742,mollieobrien.com/lodge/security.html,mollieobrien.com,,/lodge/security.html,
4521,kaptest.com/press/tips-shop,kaptest.com,,/press/tips-shop,


## 3.2 Level 1 Features: Lexical Counts

For each URL the system computes the **absolute lengths** of the full string, the isolated domain (network location) and the path:

| Feature | Definition |
|---|---|
| `url_length` | `len()` of the entire cleaned URL string |
| `domain_length` | `len()` of the network location |
| `path_length` | `len()` of the parsed path |


In [15]:
def level1_features(url: str, comp: dict) -> dict:
    return {
        "url_length":    len(url),
        "domain_length": len(comp["domain"]),
        "path_length":   len(comp["path"]),
    }

with ResourceProfiler("Level 1: lexical lengths (Milestone 3)", len(df), 3):
    level1_rows = [level1_features(u, c) for u, c in zip(df["url"], components)]

level1_df = pd.DataFrame(level1_rows)
print("Level 1 features extracted:", list(level1_df.columns))
level1_df.describe().round(2)

Level 1 features extracted: ['url_length', 'domain_length', 'path_length']


,url_length,domain_length,path_length
count,10000.00,10000.00,10000.00
mean,59.57,18.75,17.61
std,104.89,19.68,46.84
min,5.00,4.00,0.00
25%,15.00,10.00,1.00
50%,28.00,14.00,7.00
75%,58.00,21.00,17.00
max,2002.00,232.00,1992.00


---
# Milestone 4: Obfuscation-Aware (Level 2) Feature Engineering Module

Level 2 asks *how a URL is mathematically structured* rather than *what it looks like*. Whatever the obfuscation technique (random subdomains, hex/percent-encoding, padded tracking tokens, nested redirects), the result carries measurable statistical signatures: elevated entropy, uneven token lengths, abnormal digit/symbol density, inflated structural depth and encoded payloads in the query. Level 2 contributes **10 features**, implemented below in the same order as the proposal's *Feature Extraction Execution* procedures.

## 4.1 Lexical Counts: Structural Complexity (3 features)

| Feature | Procedure |
|---|---|
| `subdomain_count` | `tldextract` isolates the subdomain string; it is split on periods and non-empty labels are counted |
| `path_depth` | The parsed path is split on `/`; non-empty segments are counted |
| `query_params_count` | The query is parsed into key-value pairs (`parse_qsl`, blank values kept) and every pair is counted |

These catch evasion that stays *unremarkable at character level*, e.g. `secure.login.update.verify.bank-example.com`, where short common words keep entropy and token variance low but the subdomain chain is abnormally deep.


In [16]:
def subdomain_count(subdomain: str) -> int:
    return len([label for label in subdomain.split(".") if label])

def path_depth(path: str) -> int:
    return len([seg for seg in path.split("/") if seg])

def query_params_count(query: str) -> int:
    return len(parse_qsl(query, keep_blank_values=True))

print(subdomain_count("secure.login.update.verify"), path_depth("/a/b/c/"), query_params_count("a=1&b=&c"))

4 3 3


## 4.2 Ratio Generation (2 features)

A counting loop evaluates **every character** of the URL, tallying digits and non-alphanumeric symbols, then divides each tally by the total URL length (Equation 33):

$$\text{Ratio}=\frac{\text{Target}}{\text{Total}}$$

- `digit_ratio` = digits / URL length. Hexadecimal injection and random strings push this well above the ~0.3 level that natural-language strings rarely reach.
- `special_char_ratio` = non-alphanumeric characters / URL length. Percent-encoding (`%`, repeated delimiters) inflates it.


In [17]:
def char_ratios(url: str):
    digits = special = 0
    for ch in url:                       # specialised counting loop over every character
        if ch in "0123456789":
            digits += 1
        elif not ch.isalnum():
            special += 1
    total = len(url)
    return (digits / total, special / total) if total else (0.0, 0.0)

print(char_ratios("a1b2c3.com/%2F"))

(0.2857142857142857, 0.21428571428571427)


## 4.3 Complexity Derivation: Token Variance (1 feature)

The URL is split with the regular expression `[^a-zA-Z0-9]` into purely alphanumeric **tokens**; the length of each is recorded and the **sample variance** (Equation 32) is taken:

$$s^2=\frac{\sum_{i=1}^{n}(x_i-\bar{x})^2}{n-1}$$

Legitimate URLs tend to have tokens of similar length. Mixing short deceptive labels with long encoded strings produces a large spread. `statistics.variance` needs at least two tokens, so a URL with fewer yields `0.0`.


In [18]:
TOKEN_SPLIT = re.compile(r"[^a-zA-Z0-9]")

def token_variance(url: str) -> float:
    lengths = [len(tok) for tok in TOKEN_SPLIT.split(url) if tok]
    return statistics.variance(lengths) if len(lengths) > 1 else 0.0

print(token_variance("login.example.com/a"), token_variance("x.example.com/7f3a9c1e5b2d48a6c0de"))

6.666666666666667 72.91666666666667


## 4.4 Entropy Calculation: Domain Entropy (1 feature)

Shannon entropy (Equation 31) is computed on the isolated domain string by tallying the frequency of every distinct character:

$$H(X)=-\sum_{i=0}^{N-1}p_i\log_2 p_i,\qquad p_i=\frac{\text{count of character } i}{\text{string length}}$$

Natural-language labels draw on a small, skewed character set (low entropy); algorithmically generated labels spread over more characters more evenly (high entropy). Because the domain here is the full network location, randomly generated **subdomains** are included.

> **Note on the proposal's worked example.** Section 3.5.1 quotes ~2.58 bits for `secure`. That figure equals log₂(6), the *maximum* for a 6-character string with all-distinct characters. Since `secure` repeats the letter `e`, its true entropy is ≈ **2.25** bits (computed below). The `x4kq9zr1` example (8 distinct characters) is exactly 3.0 bits, as stated. The proposal text should be corrected.


In [19]:
def shannon_entropy(s: str) -> float:
    if not s:
        return 0.0
    n = len(s)
    return 0.0 - sum((c / n) * math.log2(c / n) for c in Counter(s).values())

for s in ["secure", "x4kq9zr1", "aaaa", "paypal.com"]:
    print(f"H({s!r}) = {shannon_entropy(s):.4f} bits")

H('secure') = 2.2516 bits
H('x4kq9zr1') = 3.0000 bits
H('aaaa') = 0.0000 bits
H('paypal.com') = 2.9219 bits


## 4.5 Query Analysis with Unfurl (3 features)

Attackers frequently smuggle a **secondary malicious URL inside a query parameter**, in plaintext, percent-encoded, double-encoded or base64-encoded, using redirect-style keys. A standard parser sees one opaque token, so the isolated **query component** is passed through **Unfurl**, which parses it into key/value pairs and *recursively decodes* percent-encoded and base64-encoded content (child nodes under each pair). The decoded structure is then evaluated:

| Feature | Computed from | Rule |
|---|---|---|
| `redirect_key_flag` | Decoded query **keys** | `1` if any key equals one of `redirect, url, next, continue, dest, return, goto, target` |
| `nested_url_flag` | Decoded query **values** (and everything Unfurl decoded beneath them) | `1` if any contains an embedded URL pattern: `http`, `https`, `www.`, or their encoded equivalents |
| `query_encoding_ratio` | The **raw, pre-decoding** query string | number of percent-encoded sequences (`%XX`) ÷ total query length |

Unfurl is run with **remote lookups disabled**, so extraction is offline and reproducible. An empty query short-circuits to `0 / 0 / 0.0`. If Unfurl is not installed, a built-in recursive decoder (repeated percent-decoding + base64 attempts) is used instead and the active backend is reported below.


In [20]:
import base64
from importlib import metadata

REDIRECT_TERMS = {"redirect", "url", "next", "continue", "dest", "return", "goto", "target"}
URL_PATTERN    = re.compile(r"https?(?::|%3a)(?://|%2f%2f)|www(?:\.|%2e)", re.IGNORECASE)
PERCENT_SEQ    = re.compile(r"%[0-9a-fA-F]{2}")
B64_CANDIDATE  = re.compile(r"^[A-Za-z0-9+/_-]{8,}={0,2}$")

UNFURL_ERRORS = 0
_unfurl = unfurl_core.Unfurl(remote_lookups=False) if UNFURL_AVAILABLE else None

def unfurl_decode_query(query: str):
    # Run Unfurl on the bare query component. Returns [(decoded_key, [decoded value + all content decoded beneath it])].
    _unfurl.reset_graph_state()
    _unfurl.add_to_queue(data_type="url.query", key=None, value=query)
    _unfurl.parse_queue()
    nodes = _unfurl.nodes
    children = defaultdict(list)
    for nid, node in nodes.items():
        children[node.parent_id].append(nid)

    pairs = []
    for nid, node in nodes.items():
        if node.data_type != "url.query.pair":
            continue
        payloads, stack = [str(node.value)], list(children[nid])
        while stack:                                   # walk everything Unfurl decoded under this pair
            cid = stack.pop()
            child = nodes[cid]
            if child.data_type != "descriptor":        # skip Unfurl's human-readable annotations
                payloads.append(str(child.value))
            stack.extend(children[cid])
        pairs.append((str(node.key), payloads))
    _unfurl.reset_graph_state()
    return pairs

def _try_base64(text: str):
    if not B64_CANDIDATE.match(text):
        return None
    try:
        out = base64.urlsafe_b64decode(text + "=" * (-len(text) % 4)).decode("utf-8")
    except Exception:
        return None
    return out if out.isprintable() else None

def fallback_decode_query(query: str):
    # Built-in substitute for Unfurl: recursive percent-decoding and base64 attempts.
    pairs = []
    for key, value in parse_qsl(query, keep_blank_values=True):    # keys/values are percent-decoded once
        payloads, cur = [value], value
        for _ in range(3):                                         # double / triple encoding
            nxt = unquote(cur)
            if nxt == cur:
                break
            payloads.append(nxt); cur = nxt
        for cand in list(payloads):
            decoded = _try_base64(cand)
            if decoded:
                payloads.append(decoded)
        pairs.append((key, payloads))
    return pairs

def query_features(query: str) -> dict:
    if not query:
        return {"redirect_key_flag": 0, "nested_url_flag": 0, "query_encoding_ratio": 0.0}
    global UNFURL_ERRORS
    try:
        pairs = unfurl_decode_query(query) if UNFURL_AVAILABLE else fallback_decode_query(query)
    except Exception:
        UNFURL_ERRORS += 1
        pairs = fallback_decode_query(query)
    return {
        "redirect_key_flag":    int(any(k.strip().lower() in REDIRECT_TERMS for k, _ in pairs)),
        "nested_url_flag":      int(any(URL_PATTERN.search(p) for _, vals in pairs for p in vals)),
        "query_encoding_ratio": len(PERCENT_SEQ.findall(query)) / len(query),   # raw, pre-decoding query
    }

backend = "Unfurl " + metadata.version("dfir-unfurl") if UNFURL_AVAILABLE else "built-in fallback decoder"
print("Query-analysis backend:", backend)
for q in ["redirect=https%3A%2F%2Fevil.com%2Fx&id=7f3a9c1e",       # percent-encoded nested URL + redirect key
          "q=aHR0cDovL2V2aWwuY29t",                                 # base64 of http://evil.com
          "r%65direct=%68ttp%3A%2F%2Fevil.com",                     # encoded key and encoded 'http'
          "utm_source=news&page=2"]:                                # benign tracking query
    print(f"{q[:55]:<57}", query_features(q))

Query-analysis backend: built-in fallback decoder
redirect=https%3A%2F%2Fevil.com%2Fx&id=7f3a9c1e           {'redirect_key_flag': 1, 'nested_url_flag': 1, 'query_encoding_ratio': 0.0851063829787234}
q=aHR0cDovL2V2aWwuY29t                                    {'redirect_key_flag': 0, 'nested_url_flag': 1, 'query_encoding_ratio': 0.0}
r%65direct=%68ttp%3A%2F%2Fevil.com                        {'redirect_key_flag': 1, 'nested_url_flag': 1, 'query_encoding_ratio': 0.14705882352941177}
utm_source=news&page=2                                    {'redirect_key_flag': 0, 'nested_url_flag': 0, 'query_encoding_ratio': 0.0}


## 4.6 Validation on Worked Examples

Before running at scale, the extraction functions are combined and checked against hand-computed values, including the proposal's own examples (a multi-tier subdomain URL, `x4kq9zr1`, a benign page, and an obfuscated redirect).

> **Second note on the proposal's examples.** Section 3.5.5 says `secure.login.update.verify.malicious-domain.com/a` has a subdomain count of **five**. By the proposal's own definition (labels *above the registered domain*) it is **four**: `secure`, `login`, `update`, `verify`.


In [21]:
FEATURE_COLUMNS = [
    "url_length", "domain_length", "path_length",                                  # Level 1
    "subdomain_count", "path_depth", "query_params_count",                        # Level 2: structural
    "digit_ratio", "special_char_ratio", "token_variance", "domain_entropy",      # Level 2: ratios / variance / entropy
    "redirect_key_flag", "nested_url_flag", "query_encoding_ratio",               # Level 2: query analysis
]
LEVEL1_COLUMNS = FEATURE_COLUMNS[:3]
LEVEL2_COLUMNS = FEATURE_COLUMNS[3:]

def level2_core(url: str, comp: dict) -> dict:
    digit_ratio, special_ratio = char_ratios(url)
    return {
        "subdomain_count":    subdomain_count(comp["subdomain"]),
        "path_depth":         path_depth(comp["path"]),
        "query_params_count": query_params_count(comp["query"]),
        "digit_ratio":        digit_ratio,
        "special_char_ratio": special_ratio,
        "token_variance":     token_variance(url),
        "domain_entropy":     shannon_entropy(comp["domain"]),
    }

def extract_all(url: str) -> dict:
    comp = parse_url(url)
    return {**level1_features(url, comp), **level2_core(url, comp), **query_features(comp["query"])}

demo_urls = {
    "benign page":          "example.com/products/shoes?page=2",
    "multi-tier subdomain": "secure.login.update.verify.malicious-domain.com/a",
    "random subdomain":     "x4kq9zr1.example.com/a",
    "obfuscated redirect":  "tracker.example.com/r?url=https%3A%2F%2Fevil.example.net%2Fx&id=7f3a9c1e5b2d48a6",
}
demo = pd.DataFrame({name: extract_all(u) for name, u in demo_urls.items()}).T[FEATURE_COLUMNS]

# hand-computed checks
assert extract_all(demo_urls["multi-tier subdomain"])["subdomain_count"] == 4
assert abs(shannon_entropy("x4kq9zr1") - 3.0) < 1e-9
assert abs(shannon_entropy("secure") - 2.2516) < 1e-3
assert shannon_entropy("aaaa") == 0.0
assert extract_all(demo_urls["benign page"])["query_params_count"] == 1
assert extract_all(demo_urls["benign page"])["path_depth"] == 2
assert extract_all(demo_urls["obfuscated redirect"])["redirect_key_flag"] == 1
assert extract_all(demo_urls["obfuscated redirect"])["nested_url_flag"] == 1
assert extract_all(demo_urls["benign page"])["redirect_key_flag"] == 0
assert extract_all(demo_urls["benign page"])["nested_url_flag"] == 0
print("All validation checks passed.\n")
demo.round(3)

All validation checks passed.



,url_length,domain_length,path_length,subdomain_count,path_depth,query_params_count,digit_ratio,special_char_ratio,token_variance,domain_entropy,redirect_key_flag,nested_url_flag,query_encoding_ratio
benign page,33.0,11.0,15.0,0.0,2.0,1.0,0.030,0.152,6.667,3.096,0.0,0.0,0.000
multi-tier subdomain,49.0,47.0,2.0,4.0,1.0,0.0,0.000,0.143,5.643,4.103,0.0,0.0,0.000
random subdomain,22.0,20.0,2.0,1.0,1.0,0.0,0.136,0.136,10.917,3.922,0.0,0.0,0.000
obfuscated redirect,80.0,19.0,2.0,1.0,1.0,2.0,0.162,0.162,14.797,3.471,1.0,1.0,0.069


## 4.7 Applying Level 2 to the Whole Dataset

Extraction runs iteratively over every row. It is split into two profiled phases so the resource cost of the **Unfurl-based query analysis** can be read separately from the pure pandas / NumPy / standard-library metrics:

- **Level 2a**: structural counts, ratios, token variance, domain entropy (7 features)
- **Level 2b**: Unfurl query analysis (3 features); Unfurl is only invoked for URLs that actually carry a query string


In [22]:
with ResourceProfiler("Level 2a: counts, ratios, variance, entropy (Milestone 4)", len(df), 7):
    level2a_rows = [level2_core(u, c) for u, c in zip(df["url"], components)]

level2a_df = pd.DataFrame(level2a_rows)
level2a_df.describe().round(3)

,subdomain_count,path_depth,query_params_count,digit_ratio,special_char_ratio,token_variance,domain_entropy
count,10000.000,10000.000,10000.000,10000.000,10000.000,10000.000,10000.000
mean,0.797,1.217,0.637,0.073,0.134,122.701,3.281
std,3.233,1.411,1.937,0.110,0.049,2027.668,0.488
min,0.000,0.000,0.000,0.000,0.011,0.000,1.522
25%,0.000,0.000,0.000,0.000,0.100,4.500,2.948
50%,0.000,1.000,0.000,0.000,0.130,9.691,3.278
75%,1.000,2.000,0.000,0.111,0.161,32.000,3.614
max,41.000,21.000,52.000,0.800,0.411,84457.884,4.844


In [23]:
n_with_query = sum(bool(c["query"]) for c in components)
print(f"URLs with a query string: {n_with_query:,} of {len(components):,}  (only these are sent through "
      f"{'Unfurl' if UNFURL_AVAILABLE else 'the fallback decoder'})")

with ResourceProfiler("Level 2b: Unfurl query analysis (Milestone 4)", len(df), 3):
    level2b_rows = [query_features(c["query"]) for c in components]

level2b_df = pd.DataFrame(level2b_rows)
print("Unfurl errors handled by fallback:", UNFURL_ERRORS)
level2b_df.describe().round(4)

URLs with a query string: 2,211 of 10,000  (only these are sent through the fallback decoder)
Unfurl errors handled by fallback: 0


,redirect_key_flag,nested_url_flag,query_encoding_ratio
count,10000.0000,10000.0000,10000.0000
mean,0.0079,0.0233,0.0040
std,0.0885,0.1509,0.0207
min,0.0000,0.0000,0.0000
25%,0.0000,0.0000,0.0000
50%,0.0000,0.0000,0.0000
75%,0.0000,0.0000,0.0000
max,1.0000,1.0000,0.3226


## 5. Computational Resource Profile (`psutil`)

The table below reports the logged resource metrics for each extraction phase. *Level 2 total* and *All 13 features* are derived rows: times and CPU time are summed (the phases ran sequentially), peak memory is the maximum observed. Lexical parsing is shared by both levels and is reported separately, so when comparing the three feature configurations of the study (traditional / obfuscation-aware / combined), add the parsing cost to whichever configuration is being priced.


In [24]:
prof = pd.DataFrame(RESOURCE_LOG)
lvl2 = prof[prof["phase"].str.startswith("Level 2")]
extra = pd.DataFrame([
    {"phase": "Level 2 total (2a + 2b)", "features": int(lvl2["features"].sum()),
     "start": lvl2["start"].iloc[0], "end": lvl2["end"].iloc[-1],
     "wall_s": lvl2["wall_s"].sum(), "cpu_time_s": lvl2["cpu_time_s"].sum(),
     "cpu_util_%": lvl2["cpu_time_s"].sum() / lvl2["wall_s"].sum() * 100,
     "peak_rss_mb": lvl2["peak_rss_mb"].max(), "delta_rss_mb": lvl2["delta_rss_mb"].max(),
     "ms_per_url": lvl2["ms_per_url"].sum()},
    {"phase": "ALL 13 features (parsing + Level 1 + Level 2)", "features": 13,
     "start": prof["start"].iloc[0], "end": prof["end"].iloc[-1],
     "wall_s": prof["wall_s"].sum(), "cpu_time_s": prof["cpu_time_s"].sum(),
     "cpu_util_%": prof["cpu_time_s"].sum() / prof["wall_s"].sum() * 100,
     "peak_rss_mb": prof["peak_rss_mb"].max(), "delta_rss_mb": prof["delta_rss_mb"].max(),
     "ms_per_url": prof["ms_per_url"].sum()},
])
profile_table = pd.concat([prof, extra], ignore_index=True)
profile_table.round({"wall_s": 3, "cpu_time_s": 3, "cpu_util_%": 1, "peak_rss_mb": 1,
                     "delta_rss_mb": 2, "ms_per_url": 4})

,phase,features,start,end,wall_s,cpu_time_s,cpu_util_%,peak_rss_mb,delta_rss_mb,ms_per_url
0,Lexical parsing (shared by Levels 1 & 2),0,09:11:19,09:11:19,0.191,0.17,88.5,208.0,0.02,0.0191
1,Level 1: lexical lengths (Milestone 3),3,09:11:19,09:11:19,0.037,0.03,78.7,208.0,0.00,0.0037
2,"Level 2a: counts, ratios, variance, entropy (Milestone 4)",7,09:11:20,09:11:20,0.697,0.67,95.5,208.0,0.02,0.0697
3,Level 2b: Unfurl query analysis (Milestone 4),3,09:11:20,09:11:21,0.142,0.11,73.9,209.8,1.00,0.0142
4,Level 2 total (2a + 2b),10,09:11:20,09:11:21,0.839,0.78,93.0,209.8,1.00,0.0839
5,ALL 13 features (parsing + Level 1 + Level 2),13,09:11:19,09:11:21,1.067,0.98,91.8,209.8,1.00,0.1067


## 6. Sanity Checks

Quick checks that the extracted features behave as the proposal's hypotheses predict. Obfuscation-aware features (entropy, digit ratio, token variance, structural depth, query-level flags) should differ between classes, and nothing should be missing, negative or out of range.


In [25]:
features_df = pd.concat([level1_df, level2a_df, level2b_df], axis=1)[FEATURE_COLUMNS]

assert len(features_df) == len(df), "row misalignment"
assert features_df.notna().all().all(), "missing feature values"
assert (features_df >= 0).all().all(), "negative feature value"
assert features_df[["digit_ratio", "special_char_ratio"]].le(1).all().all(), "ratio above 1"
assert features_df[["redirect_key_flag", "nested_url_flag"]].isin([0, 1]).all().all(), "flags must be binary"
print("Integrity checks passed: aligned, no NaN, no negatives, ratios in [0,1], flags binary.\n")

tmp = features_df.assign(label=df["label"].values)
comparison = tmp.groupby("label")[FEATURE_COLUMNS].mean().T
comparison.columns = ["benign (0) mean", "malicious (1) mean"]
comparison.insert(0, "level", [1] * 3 + [2] * 10)
comparison.round(4)

Integrity checks passed: aligned, no NaN, no negatives, ratios in [0,1], flags binary.



,level,benign (0) mean,malicious (1) mean
url_length,1,68.9232,50.2240
domain_length,1,13.1998,24.3050
path_length,1,19.0040,16.2232
subdomain_count,2,0.0892,1.5044
path_depth,2,1.2608,1.1732
query_params_count,2,0.9210,0.3528
digit_ratio,2,0.0575,0.0879
special_char_ratio,2,0.1309,0.1381
token_variance,2,162.7466,82.6556
domain_entropy,2,3.1780,3.3834


## 7. Final Output: Dataset Augmented with All 13 Features

The 13 features are merged back into the pandas data frame, **aligned with the original binary class labels** (row order is untouched), and saved for the next stage (scaling and the 80/20 split). The final cell displays the augmented dataset: `url`, `label`, plus the 3 Level 1 and 10 Level 2 features.


In [26]:
OUTPUT_PATH = "Final Dataset_Features.csv"

df_features = pd.concat([df[["url", "label"]], features_df], axis=1)

assert list(df_features.columns) == ["url", "label"] + FEATURE_COLUMNS
assert df_features.shape == (len(df), 15) and df_features[FEATURE_COLUMNS].notna().all().all()
df_features.to_csv(OUTPUT_PATH, index=False)

print(f"Augmented dataset: {df_features.shape[0]:,} rows x {df_features.shape[1]} columns "
      f"(url, label + {len(FEATURE_COLUMNS)} features: {len(LEVEL1_COLUMNS)} Level 1 + {len(LEVEL2_COLUMNS)} Level 2)")
print("Saved to:", OUTPUT_PATH)
df_features

Augmented dataset: 10,000 rows x 15 columns (url, label + 13 features: 3 Level 1 + 10 Level 2)
Saved to: Final Dataset_Features.csv


,url,label,url_length,domain_length,path_length,subdomain_count,path_depth,query_params_count,digit_ratio,special_char_ratio,token_variance,domain_entropy,redirect_key_flag,nested_url_flag,query_encoding_ratio
0,pkobp.mesavvoodltd.com/ca/#bhvryxn6lm5vd2lja2lacgtvynaucgw=,1,59,22,4,1,1,0,0.067797,0.101695,145.300000,3.641250,0,0,0.000000
1,builder.aws.com/?trk=8fe18e89-d22f-462a-8d9a-4fa301e62e02&sc_chann...,0,71,15,1,1,0,2,0.281690,0.169014,8.992424,3.773557,0,0,0.000000
2,mojohost.com,0,12,12,0,0,0,0,0.000000,0.083333,12.500000,2.751629,0,0,0.000000
3,filesafe.space,0,14,14,0,0,0,0,0.000000,0.071429,4.500000,3.039149,0,0,0.000000
4,jappy.com/tl?url=_x_tr_sl%3dauto%26_x_tr_tl%3dor%26_x_tr_hl%3den-u...,0,89,9,3,0,1,1,0.123596,0.280899,2.006667,2.947703,1,0,0.105263
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,mysupport460.weebly.com/,1,24,23,1,1,0,0,0.125000,0.125000,21.000000,4.001823,0,0,0.000000
9996,com.ausspl.com.aumail7.sspl.com.aumail7.sspl.com.au/mobile/front,1,64,51,13,7,2,0,0.031250,0.171875,2.992424,3.330794,0,0,0.000000
9997,suppors-metabusiness.pages.dev/,1,31,30,1,1,0,0,0.000000,0.129032,14.916667,3.764735,0,0,0.000000
9998,eventbrite.com.au/blog/,0,23,17,6,0,1,0,0.000000,0.173913,12.916667,3.572469,0,0,0.000000
